# Vector RAG Pipeline — FinanceBench (full batch run)

**Paper:** Kim et al. (2025), _GAR: Generative Answer Refinement for Financial QA_, arXiv 2503.15191
**Thesis:** Vector RAG vs. Vectorless RAG vs. Long-Context LLMs on FinanceBench

---

## What changed from the single-question version

The earlier two-notebook setup (`vector_rag_pipeline.ipynb` for Colab-only indexing,
`vector_rag_pipeline_local.ipynb` for the rest) existed because Stella 1.5B couldn't
load locally, so query vectors had to be embedded in Colab and hand-carried to a
local kernel one `.npy` file at a time. Now that this notebook runs directly against
a Colab-backed kernel from VS Code, that split is gone — everything below runs in one
kernel session, and there's no more manual file hand-off.

All the actual logic (chunking, indexing, retrieval, generation) now lives in
importable modules under `pipelines/vector_rag/`, not in these cells — this notebook
just calls them in order. That's what makes a 150-question × 84-document batch run
practical instead of copy-pasting cells 150 times: the same functions that were
proven correct on one question are reused for all of them.

```
Stage 0  Setup            — Drive mount, repo clone/pull, API keys, cost tracker
Stage 1  Load data        — 150 questions, 84 unique documents
Stage 2  Load models      — Stella tokenizer + embedding model (once)
Stage 3  Index ALL docs   — chunk + embed every document (resumable, GPU)
Stage 4  Embed ALL queries— expand + embed every question's query (resumable, GPU)
Stage 5  Run ALL questions— hybrid retrieve → rerank → select → generate → score
Stage 6  Summarize        — answer quality, retrieval Recall/MRR, latency, tokens
```

### Resumability — important for a run this long

Stages 3-5 are each **resumable**: every document / query / question that already has
saved output on disk is skipped, and only new work is done. If the Colab session
disconnects partway through (indexing all 84 docs is GPU-bound and can take a few
hours), just re-run the same cell — it picks up where it left off instead of
restarting from zero. Failures are caught per-item (one bad PDF or one malformed API
response doesn't abort the whole batch) and logged to `.log` files next to the
output, which are worth checking after a run finishes.

### Known scope decisions for this pass (see chat for the reasoning)
- **Gemini only.** DeepSeek V4 isn't wired in yet — `generation_model` is passed
  explicitly everywhere so adding it later is a config change, not a rewrite.
- **One run per question**, not the "median of several runs" the project brief
  specifies for latency. Fine for getting answer-quality and retrieval numbers now;
  revisit before those numbers go in the thesis.


---
## Stage 0 — Setup

Drive mount, repo clone/pull, API keys, cost tracker. Same as the original Colab notebook's Stage 0.

In [7]:
import os, sys, json, time, textwrap
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
if not (REPO_ROOT / "data" / "financebench_open_source.jsonl").exists():
    print(f"Repo not found at {REPO_ROOT} — cloning ...")
    !git clone https://github.com/shaliqsv/financebench-rag-thesis.git "{REPO_ROOT}"
else:
    print(f"Repo already present at {REPO_ROOT} — pulling latest ...")
    !git -C "{REPO_ROOT}" pull

DATA_DIR  = REPO_ROOT / "data"
PDF_DIR   = REPO_ROOT / "pdfs"
INDEX_DIR = REPO_ROOT / "experiments" / "results" / "vector_rag_index"
QUERIES_DIR = INDEX_DIR / "queries"
RESULTS_PATH = REPO_ROOT / "experiments" / "results" / "vector_rag_results.jsonl"

sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT: {REPO_ROOT}")

load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY   = os.getenv("GOOGLE_API_KEY", "")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
GROQ_API_KEY     = os.getenv("GROQ_API_KEY", "")
VOYAGE_API_KEY   = os.getenv("VOYAGE_API_KEY", "")

print("\nAPI keys:")
for name, val in [("GOOGLE_API_KEY", GOOGLE_API_KEY), ("DEEPSEEK_API_KEY", DEEPSEEK_API_KEY),
                  ("GROQ_API_KEY", GROQ_API_KEY), ("VOYAGE_API_KEY", VOYAGE_API_KEY)]:
    print(f"  {name:<20}: {'ok' if val else 'MISSING - fill in .env'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already present at /content/drive/MyDrive/financebench_project — pulling latest ...
You are not currently on a branch.
Please specify which branch you want to merge with.
See git-pull(1) for details.

    git pull <remote> <branch>

REPO_ROOT: /content/drive/MyDrive/financebench_project

API keys:
  GOOGLE_API_KEY      : MISSING - fill in .env
  DEEPSEEK_API_KEY    : MISSING - fill in .env
  GROQ_API_KEY        : MISSING - fill in .env
  VOYAGE_API_KEY      : MISSING - fill in .env


In [8]:
%pip install -q pymupdf4llm tiktoken sentence-transformers rank_bm25 pyarrow "transformers==4.51.3" "sentence-transformers==3.3.1" google-genai voyageai groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 89.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 68.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 21.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 54.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depen

In [9]:
from evaluation.cost_tracker import CostTracker
from groq import Groq
from google import genai
import voyageai

cost_tracker = CostTracker(REPO_ROOT / "experiments" / "results" / "vector_rag_costs.jsonl")
print(f"Logging costs to {cost_tracker.log_path}")

GENERATION_MODEL = "gemini-3.5-flash"   # gemini-2.5-flash was deprecated for new API keys/projects
JUDGE_MODEL = "openai/gpt-oss-120b"     # via Groq free tier — see CLAUDE.md for why

genai_client   = genai.Client(api_key=GOOGLE_API_KEY)
voyage_client  = voyageai.Client(api_key=VOYAGE_API_KEY)
judge_client   = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

print(f"Generation model: {GENERATION_MODEL}")
print(f"Judge model:      {JUDGE_MODEL}")

Logging costs to /content/drive/MyDrive/financebench_project/experiments/results/vector_rag_costs.jsonl


ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

---
## Stage 1 — Load FinanceBench data

All 150 questions across 84 unique source documents (some documents have multiple questions).

In [ ]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

doc_names = sorted(df.doc_name.unique())

print(f"Total questions : {len(df)}")
print(f"Unique documents: {len(doc_names)}")
df[["financebench_id", "doc_name", "question"]].head(3)

---
## Stage 2 — Load Stella (tokenizer + embedding model)

Loaded once here and passed into every batch call below, instead of being reloaded
per document/query — reloading a 1.5B-parameter model 84+150 times would dominate
the runtime for no benefit.

In [ ]:
from transformers import AutoTokenizer

STELLA_MODEL = "NovaSearch/stella_en_1.5B_v5"
print(f"Loading tokenizer for {STELLA_MODEL} ...")
_stella_tokenizer = AutoTokenizer.from_pretrained(STELLA_MODEL, trust_remote_code=True)

def count_tokens(text: str) -> int:
    return len(_stella_tokenizer.encode(text, add_special_tokens=False))

print("Tokenizer ready")

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("No GPU detected — indexing 84 documents on CPU will be very slow. "
          "In Colab: Runtime -> Change runtime type -> T4 GPU.")

t0 = time.time()
embed_model = SentenceTransformer(
    STELLA_MODEL,
    trust_remote_code=True,
    device=device,
    config_kwargs={"use_memory_efficient_attention": False, "unpad_inputs": False},
)
print(f"Loaded in {time.time() - t0:.1f}s — embedding dim {embed_model.get_sentence_embedding_dimension()}")

---
## Stage 3 — Index every document

Chunk (page-bounded, ≤512 tokens) + embed (Stella) every one of the 84 unique
documents, saving `{doc_name}_chunks.parquet` and `{doc_name}_dense.npy` to
`INDEX_DIR`. **Resumable** — a document already indexed is skipped, so re-running
this cell after an interruption only does the remaining work.

This is the slow part of the whole pipeline: on the first test document (160 pages),
parsing took ~150s and embedding ~145s, so budget roughly 5 minutes/document ×
84 documents ≈ several hours for a from-scratch run. Safe to leave running
unattended and re-run if the session drops.

In [ ]:
from pipelines.vector_rag.indexing import index_all_documents

index_results = index_all_documents(
    doc_names=doc_names,
    pdf_dir=PDF_DIR,
    embed_model=embed_model,
    count_tokens=count_tokens,
    index_dir=INDEX_DIR,
)

---
## Stage 4 — Expand + embed every question's query

For each of the 150 questions: ask Gemini to expand the question into a retrieval-
friendly query (Stage 4 from the single-question notebook), then embed that expanded
query with Stella. Saves `{financebench_id}__expanded.npy` / `.json` to
`QUERIES_DIR`. **Resumable** the same way as Stage 3.

This still needs the GPU-backed Stella model, which is why it runs here rather than
in Stage 5 below.

In [ ]:
from pipelines.vector_rag.batch_embed_queries import embed_all_queries

query_embed_results = embed_all_queries(
    df=df,
    genai_client=genai_client,
    embed_model=embed_model,
    expansion_model=GENERATION_MODEL,
    queries_dir=QUERIES_DIR,
    cost_tracker=cost_tracker,
)

---
## Stage 5 — Run every question end to end

For each question with both a document index (Stage 3) and a query embedding
(Stage 4) available: hybrid retrieve (top 20) → Voyage rerank (top 10) → selection
agent → generate → score against the gold answer. Appends one result row to
`RESULTS_PATH` per question as soon as it finishes, and **skips any
`financebench_id` already present there** — safe to re-run after an interruption,
and safe to run again later if Stage 3/4 produce more indexed docs or embedded
queries.

This stage doesn't need the GPU — if it's ever more convenient, `run_all_questions`
can be called from a plain CPU kernel as long as `INDEX_DIR`/`QUERIES_DIR` are
reachable (e.g. after `git pull` on the Drive clone).

In [ ]:
from pipelines.vector_rag.batch_run import run_all_questions

run_results = run_all_questions(
    df=df,
    index_dir=INDEX_DIR,
    queries_dir=QUERIES_DIR,
    out_path=RESULTS_PATH,
    genai_client=genai_client,
    voyage_client=voyage_client,
    judge_client=judge_client,
    generation_model=GENERATION_MODEL,
    judge_model=JUDGE_MODEL,
    cost_tracker=cost_tracker,
)

---
## Stage 6 — Summarize

Answer-quality breakdown, retrieval Recall@k/MRR@k (hybrid vs. reranked), latency, and token totals per stage, computed over whatever's in `RESULTS_PATH` so far — doesn't require the full 150 to be done.

In [ ]:
from pipelines.vector_rag.summarize import summarize_results, print_summary

summary = summarize_results(RESULTS_PATH, cost_log_path=cost_tracker.log_path)
print_summary(summary)

### Next steps
1. Check `INDEX_DIR/indexing_errors.log`, `QUERIES_DIR/embedding_errors.log`, and
   `RESULTS_PATH.parent/run_errors.log` for anything that failed and needs a
   second look.
2. Fill in `PRICING_PER_MILLION_TOKENS` in `evaluation/cost_tracker.py` with
   current published rates — token counts are logged throughout, but `cost_usd`
   stays `None` until those are filled in.
3. Once this is stable, wire in DeepSeek V4 as a second `generation_model` and
   re-run Stage 5 with it (Stage 3/4's indexes and query embeddings are model-
   independent and don't need to be redone).
4. Revisit the "one run per question" latency simplification if the thesis needs
   median-of-N latency per the project brief.